# Fine Tunning do BART

## 1. Instalar dependências (Colab / VSCode)

In [1]:
pip install -qq transformers datasets accelerate transformers[torch]

In [2]:
import transformers
print(transformers.__version__)

5.0.0


## 2. Carregar dataset

In [3]:
import json
from datasets import Dataset

In [4]:
!wget https://raw.githubusercontent.com/camazlucas/Fine-Tuning-BART-Feat-Paths/refs/heads/main/Model/BART%20Relational%20Paths%20Fine-Tunning/dataset_train_relation.json

--2026-05-05 17:05:45--  https://raw.githubusercontent.com/camazlucas/Fine-Tuning-BART-Feat-Paths/refs/heads/main/Model/BART%20Relational%20Paths%20Fine-Tunning/dataset_train_relation.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9649265 (9.2M) [text/plain]
Saving to: ‘dataset_train_relation.json’

dataset_train_relat 100%[===================>]   9.20M  --.-KB/s    in 0.02s   

2026-05-05 17:05:46 (377 MB/s) - ‘dataset_train_relation.json’ saved [9649265/9649265]



In [5]:
!wget https://raw.githubusercontent.com/camazlucas/Fine-Tuning-BART-Feat-Paths/refs/heads/main/Model/BART%20Relational%20Paths%20Fine-Tunning/dataset_valid_relation.json

--2026-05-05 17:05:53--  https://raw.githubusercontent.com/camazlucas/Fine-Tuning-BART-Feat-Paths/refs/heads/main/Model/BART%20Relational%20Paths%20Fine-Tunning/dataset_valid_relation.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2277818 (2.2M) [text/plain]
Saving to: ‘dataset_valid_relation.json’

dataset_valid_relat 100%[===================>]   2.17M  --.-KB/s    in 0.01s   

2026-05-05 17:05:53 (221 MB/s) - ‘dataset_valid_relation.json’ saved [2277818/2277818]



### 2.1 Com Divisao em Treino e Validacao

In [ ]:


with open("dataset_final.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

print(dataset[0])

{'input': 'what genres are the films written by Frederick Hazlitt Brennan in', 'output': 'Frederick Hazlitt Brennan -> written_by_inv -> A Guy Named Joe -> has_genre -> War'}


In [6]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

### 2.2 Carregar DATASET com Treino e Validacao pré Definidas

In [8]:
with open("/content/dataset_train_relation.json", "r", encoding="utf-8") as f:
    data = json.load(f)

train_dataset = Dataset.from_list(data)

print(train_dataset[0])

{'input': 'the films written by Jim Reardon were directed by who', 'paths': ['written_by_inv -> written_by', 'written_by_inv -> directed_by']}


In [9]:
with open("dataset_valid_relation.json", "r", encoding="utf-8") as f:
    data = json.load(f)

val_dataset = Dataset.from_list(data)

print(val_dataset[0])

{'input': 'what is a movie written by Michel Deville', 'paths': ['directed_by_inv', 'written_by_inv']}


## 4. Tokenização

In [ ]:
from transformers import BartTokenizer

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

MAX_INPUT = 64
MAX_OUTPUT = 128

def preprocess(example):
    model_input = tokenizer(
        example["input"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        example["output"],
        max_length=MAX_OUTPUT,
        truncation=True,
        padding="max_length"
    )

    model_input["labels"] = labels["input_ids"]
    return model_input

train_dataset = train_dataset.map(preprocess, batched=False)
val_dataset = val_dataset.map(preprocess, batched=False)

In [11]:
from transformers import BartTokenizer

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

MAX_INPUT = 64
MAX_OUTPUT = 128

def preprocess(example):
    model_input = tokenizer(
        example["input"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )
    
    # 🔥 usa paths em vez de output
    labels_text = " <SEP> ".join(example["paths"])

    labels = tokenizer(
        labels_text,
        max_length=MAX_OUTPUT,
        truncation=True,
        padding="max_length"
    )

    model_input["labels"] = labels["input_ids"]
    return model_input

train_dataset = train_dataset.map(preprocess, batched=False)
val_dataset = val_dataset.map(preprocess, batched=False)

Map:   0%|          | 0/58454 [00:00<?, ? examples/s]

Map:   0%|          | 0/15061 [00:00<?, ? examples/s]

## 5. Carregar modelo

In [12]:
from transformers import BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

## 6. Configuração de treino

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=True,  # usar GPU
    logging_steps=100
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 7. Trainer

In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

## 8. Treinar

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.013881,0.018157
2,0.012083,0.019736
3,0.010691,0.020869


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21921, training_loss=0.029227349504568102, metrics={'train_runtime': 1368.195, 'train_samples_per_second': 128.17, 'train_steps_per_second': 16.022, 'total_flos': 6682787799367680.0, 'train_loss': 0.029227349504568102, 'epoch': 3.0})

## 9. Salvar modelo

In [16]:
trainer.save_model("bart_paths_model")
tokenizer.save_pretrained("bart_paths_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bart_paths_model/tokenizer_config.json', 'bart_paths_model/tokenizer.json')

## 10. Teste rápido

In [17]:
def predict(question):
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=128)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(predict("who directed movies written by Quentin Tarantino"))

written_by_inv -> directed_by


# Download do Modelo

In [18]:
!zip -r bart_paths_model.zip bart_paths_model

  adding: bart_paths_model/ (stored 0%)
  adding: bart_paths_model/training_args.bin (deflated 53%)
  adding: bart_paths_model/config.json (deflated 64%)
  adding: bart_paths_model/generation_config.json (deflated 47%)
  adding: bart_paths_model/tokenizer.json (deflated 82%)
  adding: bart_paths_model/tokenizer_config.json (deflated 50%)
  adding: bart_paths_model/model.safetensors (deflated 8%)


In [19]:
from google.colab import files
files.download("bart_paths_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
!zip -r bart_paths_model.zip bart_paths_model
!split -b 100M bart_paths_model.zip part_

updating: bart_paths_model/ (stored 0%)
updating: bart_paths_model/training_args.bin (deflated 53%)
updating: bart_paths_model/config.json (deflated 64%)
updating: bart_paths_model/generation_config.json (deflated 47%)
updating: bart_paths_model/tokenizer.json (deflated 82%)
updating: bart_paths_model/tokenizer_config.json (deflated 50%)
updating: bart_paths_model/model.safetensors (deflated 8%)
